# Homework 1 · Q7：一步一步计算线性回归权重

从上到下运行单元格（Shift + Enter），内核选择项目 .venv 的 Python 3.11。
先用 Pandas 筛选数据，再用 NumPy 进行矩阵运算。各步骤已保留运行结果。

## 准备：读取 CSV

pd.read_csv() 返回 DataFrame。下面从当前工作目录向上查找课程数据，因此从项目根目录或此 Notebook 所在目录启动都可以。

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

relative_path = Path("cohorts/2026/data/car_fuel_efficiency_2026.csv")
data_path = next(
    (folder / relative_path for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / relative_path).is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError("Result: ")

df = pd.read_csv(data_path)
print("Result: ", data_path)
print(df.head())

Result:  C:\Users\yanji\Projects\machine-learning-zoomcamp\cohorts\2026\data\car_fuel_efficiency_2026.csv
   model_year  origin  ... acceleration fuel_efficiency_mpg
0        2006  Europe  ...          NaN                31.9
1        2008  Europe  ...          NaN                31.3
2        1996    Asia  ...         17.3                27.5
3        1989  Europe  ...         18.7                28.5
4        1994     USA  ...         17.5                31.0

[5 rows x 11 columns]


## 1. 筛选所有来自亚洲的汽车

地区列是 origin。df["origin"] == "Asia" 对每行返回 True 或 False；外层 df[...] 只保留 True 对应的行。

In [2]:
asia_cars = df[df["origin"] == "Asia"]
print("Result: ", len(asia_cars))
print(asia_cars.head())

Result:  2994
    model_year origin  ... acceleration fuel_efficiency_mpg
2         1996   Asia  ...         17.3                27.5
6         1992   Asia  ...         21.7                26.8
8         1997   Asia  ...         19.5                30.9
10        1978   Asia  ...         19.5                26.8
15        2007   Asia  ...         18.0                35.4

[5 rows x 11 columns]


## 2. 只保留两列

用列名列表选取 vehicle_weight 和 model_year，因此有两层方括号。列的顺序决定 X 的列顺序和最终权重的顺序。

In [3]:
selected_columns = asia_cars[["vehicle_weight", "model_year"]]
print(selected_columns.head())

    vehicle_weight  model_year
2             4240        1996
6             4310        1992
8             4230        1997
10            4420        1978
15            3950        2007


## 3. 取前 7 行

head(7) 取筛选后的前 7 行，相当于 .iloc[:7]。先筛选亚洲汽车，再取前 7 行，不排序、不随机抽样。原来的行索引不影响计算。

In [4]:
first_7 = selected_columns.head(7)
print(first_7)
print("Result: ", first_7.shape)

    vehicle_weight  model_year
2             4240        1996
6             4310        1992
8             4230        1997
10            4420        1978
15            3950        2007
16            4750        2021
22            4450        2019
Result:  (7, 2)


## 4. 转换为 NumPy 数组 X

.to_numpy() 提取数值，不包含行索引和列名。X 的形状是 (7, 2)：每行一辆车，每列一个特征。dtype=float 将数值转换为浮点数。

In [5]:
X = first_7.to_numpy(dtype=float)
print(X)
print("Result: ", type(X))
print("Result: ", X.shape)

[[4240. 1996.]
 [4310. 1992.]
 [4230. 1997.]
 [4420. 1978.]
 [3950. 2007.]
 [4750. 2021.]
 [4450. 2019.]]
Result:  <class 'numpy.ndarray'>
Result:  (7, 2)


## 5. 计算 XTX = X.T @ X

X.T 是转置，将形状 (7, 2) 变成 (2, 7)。@ 是矩阵乘法，不要写成逐元素相乘的 *。

矩阵乘法要求内侧维度相等：(2, 7) @ (7, 2) → (2, 2)。

In [6]:
XTX = X.T @ X
print("Result: ", X.T.shape)
print("Result: ")
print(XTX)
print("Result: ", XTX.shape)

Result:  (2, 7)
Result: 
[[1.3195050e+08 6.0750580e+07]
 [6.0750580e+07 2.8041424e+07]]
Result:  (2, 2)


## 6. 对 XTX 求逆

np.linalg.inv() 计算逆矩阵。原矩阵与逆矩阵相乘应接近单位矩阵：对角线上是 1，其余位置是 0。浮点计算会有微小误差，因此用 np.allclose() 检查。

只有可逆方阵才能求逆，本题的 XTX 满足条件。

In [7]:
XTX_inv = np.linalg.inv(XTX)
print("Result: ")
print(XTX_inv)
print("Result: ", np.allclose(XTX_inv @ XTX, np.eye(2)))

Result: 
[[ 2.96830537e-06 -6.43071025e-06]
 [-6.43071025e-06  1.39675281e-05]]
Result:  True


## 7. 创建 y

np.array() 将列表转成数组。题目给定的 7 个值按顺序与 X 的 7 行一一对应。y 的形状是 (7,)，它是一维数组，不是形状 (7, 1) 的二维列向量。

In [8]:
y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])
print(y)
print("Result: ", y.shape)

[1100 1300  800  900 1000 1100 1200]
Result:  (7,)


## 8. 计算 w

先用逆矩阵乘 X.T，再乘 y：w = (XTX_inv @ X.T) @ y。

中间结果的形状是 (2, 7)，再乘 (7,) 的 y，得到 (2,) 的 w。两个权重依次对应 vehicle_weight 和 model_year。本题没有添加截距列。

In [9]:
intermediate = XTX_inv @ X.T
w = intermediate @ y
print("Result: ", intermediate.shape)
print("Result: ", w)
print("Result: ", w[0])
print("Result: ", w[1])

Result:  (2, 7)
Result:  [0.13644777 0.2327492 ]
Result:  0.1364477656156794
Result:  0.23274920343357547


## 回答作业：权重之和

w.sum() 将两个权重相加。将结果与作业选项比较。

In [10]:
print("Result: ", w.sum())

Result:  0.36919696904925486


## 回顾

本题计算 w = (X.T @ X) 的逆矩阵 @ X.T @ y，即线性回归的正规方程。

实际项目通常可使用 np.linalg.lstsq(X, y, rcond=None) 求最小二乘解，避免显式求逆；这里按照题目要求逐步实现。